In [11]:
!pip install -q pandas numpy scikit-learn mlflow fastapi uvicorn joblib pytest httpx reportlab

In [12]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from pathlib import Path

print("All libraries imported successfully!")

All libraries imported successfully!


In [13]:
os.makedirs("milk_mlop/data", exist_ok=True)
os.makedirs("milk_mlop/features", exist_ok=True)
os.makedirs("milk_mlop/models", exist_ok=True)
os.makedirs("milk_mlop/tests", exist_ok=True)
os.makedirs("milk_mlop/.github/workflows", exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [14]:
np.random.seed(42)

n = 250

data = pd.DataFrame({
    "milk_quantity_litres": np.random.uniform(20, 250, n),
    "collection_temperature_c": np.random.normal(27, 3, n),
    "transportation_duration_hours": np.random.uniform(0.5, 5, n),
    "cooling_delay_hours": np.random.uniform(0, 3, n),
    "milk_fat_percentage": np.random.uniform(3, 6, n),
    "acidity_indicator": np.random.uniform(0.10, 0.28, n),
    "container_type": np.random.choice(
        ["Steel", "Plastic", "Aluminium"], n
    ),
    "ambient_temperature_c": np.random.normal(28, 3, n)
})

# Generate rejection target using quality-related conditions
risk = (
    0.10 * (data["collection_temperature_c"] - 25)
    + 0.45 * data["transportation_duration_hours"]
    + 0.65 * data["cooling_delay_hours"]
    + 15 * (data["acidity_indicator"] - 0.16)
    + 0.08 * (data["ambient_temperature_c"] - 27)
    + 0.3 * (data["container_type"] == "Plastic")
)

probability = 1 / (1 + np.exp(-risk))

data["batch_rejected"] = np.random.binomial(1, probability)

data.to_csv(
    "milk_mlop/data/milk_v1_normal.csv",
    index=False
)

print("Version 1 dataset created!")
display(data.head())
print(data.shape)
print(data["batch_rejected"].value_counts())

Version 1 dataset created!


,milk_quantity_litres,collection_temperature_c,transportation_duration_hours,cooling_delay_hours,milk_fat_percentage,acidity_indicator,container_type,ambient_temperature_c,batch_rejected
0,106.144227,24.459619,4.380637,1.612187,5.963167,0.275656,Aluminium,27.910329,1
1,238.664290,22.455458,4.772843,2.759567,4.283319,0.160641,Steel,27.477059,1
2,188.358607,25.660455,1.161831,1.039038,4.152980,0.132781,Steel,26.825306,1
3,157.691451,29.569196,4.669644,1.040860,5.038942,0.242146,Plastic,27.954421,1
4,55.884287,27.642281,2.714523,2.212504,3.654762,0.218567,Steel,30.589301,1


(250, 9)
batch_rejected
1    232
0     18
Name: count, dtype: int64


In [15]:
np.random.seed(99)

n = 250

summer = pd.DataFrame({
    "milk_quantity_litres": np.random.uniform(20, 250, n),
    "collection_temperature_c": np.random.normal(33, 3, n),
    "transportation_duration_hours": np.random.uniform(1, 7, n),
    "cooling_delay_hours": np.random.uniform(0.5, 5, n),
    "milk_fat_percentage": np.random.uniform(3, 6, n),
    "acidity_indicator": np.random.uniform(0.10, 0.32, n),
    "container_type": np.random.choice(
        ["Steel", "Plastic", "Aluminium"], n
    ),
    "ambient_temperature_c": np.random.normal(35, 3, n)
})

risk = (
    0.10 * (summer["collection_temperature_c"] - 25)
    + 0.45 * summer["transportation_duration_hours"]
    + 0.65 * summer["cooling_delay_hours"]
    + 15 * (summer["acidity_indicator"] - 0.16)
    + 0.08 * (summer["ambient_temperature_c"] - 27)
    + 0.3 * (summer["container_type"] == "Plastic")
)

probability = 1 / (1 + np.exp(-risk))

summer["batch_rejected"] = np.random.binomial(1, probability)

summer.to_csv(
    "milk_mlop/data/milk_v2_summer.csv",
    index=False
)

print("Version 2 summer dataset created!")
display(summer.head())
print(summer.shape)
print(summer["batch_rejected"].value_counts())

Version 2 summer dataset created!


,milk_quantity_litres,collection_temperature_c,transportation_duration_hours,cooling_delay_hours,milk_fat_percentage,acidity_indicator,container_type,ambient_temperature_c,batch_rejected
0,174.624068,30.444850,5.171920,2.926293,4.420226,0.281822,Aluminium,35.711679,1
1,132.258032,27.549620,4.301314,4.114888,4.251376,0.223159,Plastic,35.142846,1
2,209.863890,32.344985,2.153667,1.727939,5.597916,0.116829,Aluminium,36.475497,1
3,27.232669,30.151025,2.008109,4.670284,3.217970,0.278539,Plastic,33.875759,1
4,205.851492,34.552693,6.596344,1.728731,3.625154,0.252275,Steel,32.646600,1


(250, 9)
batch_rejected
1    245
0      5
Name: count, dtype: int64


In [16]:
data["dataset_version"] = "Version 1 - Normal"
summer["dataset_version"] = "Version 2 - Summer"

combined = pd.concat(
    [data, summer],
    ignore_index=True
)

combined.to_csv(
    "milk_mlop/data/milk_combined.csv",
    index=False
)

print("Combined dataset created!")
print("Total records:", len(combined))
display(combined.head())

Combined dataset created!
Total records: 500


,milk_quantity_litres,collection_temperature_c,transportation_duration_hours,cooling_delay_hours,milk_fat_percentage,acidity_indicator,container_type,ambient_temperature_c,batch_rejected,dataset_version
0,106.144227,24.459619,4.380637,1.612187,5.963167,0.275656,Aluminium,27.910329,1,Version 1 - Normal
1,238.664290,22.455458,4.772843,2.759567,4.283319,0.160641,Steel,27.477059,1,Version 1 - Normal
2,188.358607,25.660455,1.161831,1.039038,4.152980,0.132781,Steel,26.825306,1,Version 1 - Normal
3,157.691451,29.569196,4.669644,1.040860,5.038942,0.242146,Plastic,27.954421,1,Version 1 - Normal
4,55.884287,27.642281,2.714523,2.212504,3.654762,0.218567,Steel,30.589301,1,Version 1 - Normal


In [17]:
features = combined.copy()

features["temperature_exposure_index"] = (
    0.45 * features["collection_temperature_c"]
    + 0.25 * features["ambient_temperature_c"]
) * (1 + features["cooling_delay_hours"] / 5)

features["transport_delay_risk"] = (
    features["transportation_duration_hours"]
    * (1 + features["cooling_delay_hours"] / 4)
)

features.to_csv(
    "milk_mlop/features/milk_engineered_features.csv",
    index=False
)

print("Feature engineering completed!")
display(features.head())

Feature engineering completed!


,milk_quantity_litres,collection_temperature_c,transportation_duration_hours,cooling_delay_hours,milk_fat_percentage,acidity_indicator,container_type,ambient_temperature_c,batch_rejected,dataset_version,temperature_exposure_index,transport_delay_risk
0,106.144227,24.459619,4.380637,1.612187,5.963167,0.275656,Aluminium,27.910329,1,Version 1 - Normal,23.783257,6.146238
1,238.664290,22.455458,4.772843,2.759567,4.283319,0.160641,Steel,27.477059,1,Version 1 - Normal,26.342520,8.065588
2,188.358607,25.660455,1.161831,1.039038,4.152980,0.132781,Steel,26.825306,1,Version 1 - Normal,22.046754,1.463627
3,157.691451,29.569196,4.669644,1.040860,5.038942,0.242146,Plastic,27.954421,1,Version 1 - Normal,24.519539,5.884755
4,55.884287,27.642281,2.714523,2.212504,3.654762,0.218567,Steel,30.589301,1,Version 1 - Normal,28.974577,4.215997


In [18]:
feature_store = pd.read_csv(
    "milk_mlop/features/milk_engineered_features.csv"
)

print("Feature store loaded successfully!")
print("Number of records:", len(feature_store))

display(feature_store.head())

Feature store loaded successfully!
Number of records: 500


,milk_quantity_litres,collection_temperature_c,transportation_duration_hours,cooling_delay_hours,milk_fat_percentage,acidity_indicator,container_type,ambient_temperature_c,batch_rejected,dataset_version,temperature_exposure_index,transport_delay_risk
0,106.144227,24.459619,4.380637,1.612187,5.963167,0.275656,Aluminium,27.910329,1,Version 1 - Normal,23.783257,6.146238
1,238.664290,22.455458,4.772843,2.759567,4.283319,0.160641,Steel,27.477059,1,Version 1 - Normal,26.342520,8.065588
2,188.358607,25.660455,1.161831,1.039038,4.152980,0.132781,Steel,26.825306,1,Version 1 - Normal,22.046754,1.463627
3,157.691451,29.569196,4.669644,1.040860,5.038942,0.242146,Plastic,27.954421,1,Version 1 - Normal,24.519539,5.884755
4,55.884287,27.642281,2.714523,2.212504,3.654762,0.218567,Steel,30.589301,1,Version 1 - Normal,28.974577,4.215997


In [19]:
X = feature_store.drop(
    columns=["batch_rejected", "dataset_version"]
)

y = feature_store["batch_rejected"]

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Preprocessing pipeline created!")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Preprocessing pipeline created!
Training records: 375
Testing records: 125


In [20]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

logistic_predictions = logistic_model.predict(X_test)

print("Logistic Regression trained successfully!")

print("Accuracy:",
      accuracy_score(y_test, logistic_predictions))

print("Precision:",
      precision_score(y_test, logistic_predictions, zero_division=0))

print("Recall:",
      recall_score(y_test, logistic_predictions, zero_division=0))

print("F1 Score:",
      f1_score(y_test, logistic_predictions, zero_division=0))

Logistic Regression trained successfully!
Accuracy: 0.808
Precision: 0.9797979797979798
Recall: 0.8151260504201681
F1 Score: 0.8899082568807339


In [21]:
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",
        random_state=42
    ))
])

random_forest_model.fit(X_train, y_train)

rf_predictions = random_forest_model.predict(X_test)

print("Random Forest trained successfully!")

print("Accuracy:",
      accuracy_score(y_test, rf_predictions))

print("Precision:",
      precision_score(y_test, rf_predictions, zero_division=0))

print("Recall:",
      recall_score(y_test, rf_predictions, zero_division=0))

print("F1 Score:",
      f1_score(y_test, rf_predictions, zero_division=0))

Random Forest trained successfully!
Accuracy: 0.952
Precision: 0.952
Recall: 1.0
F1 Score: 0.9754098360655737


In [22]:
comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, logistic_predictions),
        "Precision": precision_score(y_test, logistic_predictions, zero_division=0),
        "Recall": recall_score(y_test, logistic_predictions, zero_division=0),
        "F1 Score": f1_score(y_test, logistic_predictions, zero_division=0)
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, rf_predictions),
        "Precision": precision_score(y_test, rf_predictions, zero_division=0),
        "Recall": recall_score(y_test, rf_predictions, zero_division=0),
        "F1 Score": f1_score(y_test, rf_predictions, zero_division=0)
    }
])

display(comparison)

comparison.to_csv(
    "milk_mlop/model_comparison.csv",
    index=False
)

print("Model comparison saved!")

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.808,0.979798,0.815126,0.889908
1,Random Forest,0.952,0.952000,1.000000,0.975410


Model comparison saved!


In [23]:
if (
    comparison.iloc[1]["Recall"] > comparison.iloc[0]["Recall"]
    or (
        comparison.iloc[1]["Recall"] == comparison.iloc[0]["Recall"]
        and comparison.iloc[1]["F1 Score"] > comparison.iloc[0]["F1 Score"]
    )
):
    best_model = random_forest_model
    best_model_name = "Random Forest"
    best_predictions = rf_predictions
else:
    best_model = logistic_model
    best_model_name = "Logistic Regression"
    best_predictions = logistic_predictions

print("Best model:", best_model_name)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, best_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    best_predictions,
    zero_division=0
))

Best model: Random Forest

Confusion Matrix:
[[  0   6]
 [  0 119]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         6
           1       0.95      1.00      0.98       119

    accuracy                           0.95       125
   macro avg       0.48      0.50      0.49       125
weighted avg       0.91      0.95      0.93       125



In [24]:
joblib.dump(
    best_model,
    "milk_mlop/models/best_model.joblib"
)

print("Best model saved successfully!")

Best model saved successfully!


In [27]:
import mlflow
import mlflow.sklearn
import os

# Use an absolute SQLite database path
mlflow.set_tracking_uri("sqlite:////content/milk_mlop/mlflow.db")
mlflow.set_experiment("Milk_Batch_Rejection_Prediction")

trusted_types = [
    "numpy.dtype",
    "sklearn.compose._column_transformer._RemainderColsList",
    "sklearn.tree._tree.Tree"
]

with mlflow.start_run(run_name=best_model_name) as run:

    mlflow.log_param("model_name", best_model_name)
    mlflow.log_param("dataset", "Normal and Summer")

    row = comparison[comparison["Model"] == best_model_name].iloc[0]

    mlflow.log_metric("accuracy", float(row["Accuracy"]))
    mlflow.log_metric("precision", float(row["Precision"]))
    mlflow.log_metric("recall", float(row["Recall"]))
    mlflow.log_metric("f1_score", float(row["F1 Score"]))

    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="model",
        registered_model_name="MilkBatchRejectionModel",
        skops_trusted_types=trusted_types
    )

    run_id = run.info.run_id

print("MLflow tracking successful!")
print("Run ID:", run_id)
print("Registered model: MilkBatchRejectionModel")

MLflow tracking successful!
Run ID: 3847339b42ef4c329496aaa65f936d74
Registered model: MilkBatchRejectionModel


Successfully registered model 'MilkBatchRejectionModel'.
Created version '1' of model 'MilkBatchRejectionModel'.


In [39]:
%cd /content/milk_mlop

app_code = '''
from pathlib import Path
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

BASE_DIR = Path(__file__).resolve().parent
model = joblib.load(BASE_DIR / "models" / "best_model.joblib")

app = FastAPI(title="Milk Batch Rejection Prediction API")

class MilkBatch(BaseModel):
    milk_quantity: float = Field(gt=0)
    collection_temperature: float = Field(ge=0, le=60)
    transportation_duration: float = Field(ge=0, le=72)
    cooling_delay: float = Field(ge=0, le=1440)
    milk_fat: float = Field(ge=0, le=15)
    acidity: float = Field(ge=0, le=20)
    container_type: str
    ambient_temperature: float = Field(ge=-10, le=60)
    dataset_version: str = "summer"

@app.get("/")
def home():
    return {"message": "Milk Batch Rejection API is running"}

@app.post("/predict")
def predict(batch: MilkBatch):
    data = batch.dict()

    # Match the column names used during model training
    data["milk_quantity_litres"] = data.pop("milk_quantity")
    data["collection_temperature_c"] = data.pop("collection_temperature")
    data["transportation_duration_hours"] = data.pop("transportation_duration")
    data["cooling_delay_hours"] = data.pop("cooling_delay")
    data["milk_fat_percentage"] = data.pop("milk_fat")
    data["acidity_indicator"] = data.pop("acidity")
    data["ambient_temperature_c"] = data.pop("ambient_temperature")

    # Feature engineering
    data["temperature_exposure_index"] = (
        data["collection_temperature_c"] * data["cooling_delay_hours"]
    )
    data["transport_delay_risk"] = (
        data["transportation_duration_hours"] * data["ambient_temperature_c"]
    )

    input_df = pd.DataFrame([data])

    prediction = model.predict(input_df)[0]

    probability = None
    if hasattr(model, "predict_proba"):
        probability = float(model.predict_proba(input_df)[0][1])

    rejected = int(prediction) == 1

    return {
        "prediction": "Rejection Risk" if rejected else "Likely Accepted",
        "batch_rejected": int(rejected),
        "rejection_probability": probability
    }
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("Cell 16 completed: FastAPI application created successfully!")

/content/milk_mlop
Cell 16 completed: FastAPI application created successfully!


In [30]:
requirements = """
numpy
pandas
scikit-learn
joblib
mlflow
fastapi
uvicorn
pydantic
pytest
httpx
reportlab
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created")

requirements.txt created


In [31]:
# CELL 18: CREATE TEST FILE

test_code = '''
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

def valid_batch():
    return {
        "milk_quantity": 100,
        "collection_temperature": 5,
        "transportation_duration": 1,
        "cooling_delay": 10,
        "milk_fat": 4,
        "acidity": 6.5,
        "container_type": "stainless_steel",
        "ambient_temperature": 25,
        "dataset_version": "summer"
    }

def test_home():
    assert client.get("/").status_code == 200

def test_valid_prediction():
    response = client.post("/predict", json=valid_batch())
    assert response.status_code == 200
    assert response.json()["prediction"] in [
        "Likely Accepted", "Rejection Risk"
    ]

def test_negative_milk_quantity():
    data = valid_batch()
    data["milk_quantity"] = -10
    assert client.post("/predict", json=data).status_code == 422

def test_impossible_temperature():
    data = valid_batch()
    data["collection_temperature"] = 100
    assert client.post("/predict", json=data).status_code == 422
'''

with open("test_api.py", "w") as f:
    f.write(test_code)

print("test_api.py created")

test_api.py created


In [40]:
%cd /content/milk_mlop

!pytest -v test_api.py

/content/milk_mlop
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/milk_mlop
plugins: platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0
collected 4 items                                                              

test_api.py::test_home PASSED                                            [ 25%]
test_api.py::test_valid_prediction PASSED                                [ 50%]
test_api.py::test_negative_milk_quantity PASSED                          [ 75%]
test_api.py::test_impossible_temperature PASSED                          [100%]

============================== 4 passed in 1.23s ===============================


In [34]:
# CELL 20: CREATE GITHUB ACTIONS WORKFLOW

import os

os.makedirs(".github/workflows", exist_ok=True)

workflow = """
name: Milk MLOps CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run tests
        run: pytest -v test_api.py
"""

with open(".github/workflows/ci.yml", "w") as f:
    f.write(workflow)

print("GitHub Actions workflow created")

GitHub Actions workflow created


In [46]:
%cd /content/milk_mlop

!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &
!sleep 5
!cat uvicorn.log

/content/milk_mlop
INFO:     Started server process [9418]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [47]:
%cd /content/milk_mlop
!git init
!git add .
!git commit -m "Complete Milk Batch Rejection MLOps Project"
!git branch -M main

/content/milk_mlop
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/milk_mlop/.git/
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@69ec3d7c977b.(none)')


In [48]:
!git config --global user.name "Geetanjali"
!git config --global user.email "231fa04501@gmail.com"

In [49]:
%cd /content/milk_mlop
!git add .
!git commit -m "Complete Milk Batch Rejection MLOps Project"
!git branch -M main

/content/milk_mlop
[main (root-commit) 6a97fa0] Complete Milk Batch Rejection MLOps Project
 17 files changed, 1789 insertions(+)
 create mode 100644 .github/workflows/ci.yml
 create mode 100644 Milk_Batch_Rejection_MLOps_Report.pdf
 create mode 100644 README.md
 create mode 100644 __pycache__/app.cpython-313.pyc
 create mode 100644 __pycache__/test_api.cpython-313-pytest-8.4.2.pyc
 create mode 100644 app.py
 create mode 100644 data/milk_combined.csv
 create mode 100644 data/milk_v1_normal.csv
 create mode 100644 data/milk_v2_summer.csv
 create mode 100644 features/milk_engineered_features.csv
 create mode 100644 mlflow.db
 create mode 100644 model_comparison.csv
 create mode 100644 models/best_model.joblib
 create mode 100644 requirements.txt
 create mode 100644 run_evidence.json
 create mode 100644 test_api.py
 create mode 100644 uvicorn.log


In [51]:
%cd /content/milk_mlop

!git remote remove origin 2>/dev/null
!git remote add origin https://github.com/Geetanjali501/Milk-Batch-Rejection-MLOps.git

!git remote -v

/content/milk_mlop
origin	https://github.com/Geetanjali501/Milk-Batch-Rejection-MLOps.git (fetch)
origin	https://github.com/Geetanjali501/Milk-Batch-Rejection-MLOps.git (push)


In [53]:

import shutil

# Folder containing your project
folder_path = "/content/milk_mlop"

# Create ZIP file
zip_path = shutil.make_archive(
    "/content/Milk-Batch-Rejection-MLOps",
    "zip",
    folder_path
)

print("ZIP file created successfully!")
print(zip_path)


ZIP file created successfully!
/content/Milk-Batch-Rejection-MLOps.zip


In [54]:
from google.colab import files

files.download("/content/Milk-Batch-Rejection-MLOps.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>